In [1]:
import os
import sys
import time

In [2]:
import importlib
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import library.uart3_protocol as uart3_protocol
import library.aes_gcm_app as aes_gcm_app
import library.ml_kem_app as ml_kem_app
importlib.reload(uart3_protocol)
importlib.reload(aes_gcm_app)
importlib.reload(ml_kem_app)

from library.uart3_protocol import UART3Protocol
from library.aes_gcm_app import STM32AESGCM
from library.ml_kem_app import STM32MLKEM

print("Library updated / reloaded")

Library updated / reloaded


In [3]:
PORT = "/dev/cu.usbmodem1103"
BAUD = 1000000
MAX_BUFFER_SIZE = 32768
TIMEOUT = 0.01

uart = UART3Protocol(
    port = PORT,
    baud = BAUD,
    max_buffer_size = MAX_BUFFER_SIZE,
    timeout = TIMEOUT
)

uart.open()
time.sleep(0.5)

In [4]:
kem = STM32MLKEM(uart)
kem.clear()

True

In [5]:
try:
    result = kem.handshake()
except RuntimeError as e:
    print("KEM handshake failed:\n\t", e)
    print("Trying rekey...")
    kem.rekey()
    result = kem.handshake()

print("KEM handshake OK")

KEM handshake failed:
	 Expected 'READY', got 'KEM_NOT_READY'
Trying rekey...
KEM handshake OK


In [6]:
print("public_key len:", len(result["public_key"]))
print("kem_ciphertext len:", len(result["kem_ciphertext"]))
print("shared_secret_python len:", len(result["shared_secret_python"]))
print("aes_key:", result["aes_key"].hex())

public_key len: 1184
kem_ciphertext len: 1088
shared_secret_python len: 32
aes_key: fc44678e1d221983ca76ea86e51cb8ae3c080b4680caaba1d13bf0baa8b89415


In [7]:
uart.close()